# 01 — Data Load + Sanity Checks (Arena-Hard v2.0)

Pull Arena-Hard v2.0 data, run sanity checks, save preprocessed (candidate, baseline) pairs for downstream notebooks.

**Locked design (see `methodology_notes.md`):**
- Dataset: Arena-Hard v2.0 (500 **hard prompts** — filtered from 750 total; creative writing 250 excluded, different baseline)
- Candidate (held fixed at L0): `gpt-4.1-mini`
- Baseline (per-category fixed via `JUDGE_SETTINGS`): `o3-mini-2025-01-31`
- Anchor judge: `gpt-4.1` (v2.0 shipped default)
- Anchor rubric (C1): `OG_ARENA_HARD_PROMPT` — Arena-Hard's shipped default rubric
- Anchor verdict format (F1): reasoning before verdict, 5-point scale (`[[A>>B]]` / `[[A>B]]` / `[[A=B]]` / `[[B>A]]` / `[[B>>A]]`)
- Position bias: two-round bidirectional swap (Arena-Hard repo default)
- No language filter — all 500 retained (multilingual subset reserved as a future D-axis variant)

In [1]:
import json
import pandas as pd
from pathlib import Path

## Pull Arena-Hard v2.0 from Hugging Face

```bash
git lfs install
git clone https://huggingface.co/datasets/lmarena-ai/arena-hard-auto arena-hard-data
cp -r arena-hard-data/data/arena-hard-v2.0 data/
```

Also clone the repo for protocol reference (config, JUDGE_SETTINGS, gen_judgment.py):

```bash
git clone https://github.com/lmarena/arena-hard-auto.git arena-hard-repo
```

In [2]:
DATA_DIR = Path('data/arena-hard-v2.0')
REPO_DIR = Path('arena-hard-repo')

## Sanity check 1 — prompt count

In [3]:
questions = pd.read_json(DATA_DIR / 'question.jsonl', lines=True)
print(f'Total prompt count: {len(questions)}')
print(f'Category breakdown: {questions["category"].value_counts().to_dict()}')

# Filter to hard prompts (category=hard_prompt uses o3-mini-2025-01-31 baseline per JUDGE_SETTINGS).
# Excludes creative_writing (250 prompts) which uses gemini-2.0-flash-001 baseline — different scope.
hard = questions[questions['category'] == 'hard_prompt'].copy()
print(f'\nFiltered to hard prompts: {len(hard)}')
print(f'Subcategory breakdown: {hard["subcategory"].value_counts().to_dict()}')
assert len(hard) == 500, f'Expected 500 hard prompts, got {len(hard)}'
hard.head()

Total prompt count: 750
Category breakdown: {'hard_prompt': 500, 'creative_writing': 250}

Filtered to hard prompts: 500
Subcategory breakdown: {'coding': 253, 'math': 247}


,uid,category,subcategory,language,prompt
0,2edbb5f36f5b42be,hard_prompt,coding,English,Write me a zig program that solves the followi...
1,ec71c09662a64365,hard_prompt,coding,English,please write a python script that takes a .mp4...
2,d5cdf24c4e614beb,hard_prompt,coding,English,"<div style=""width: 100vh; height: 100vh;"">\n ..."
3,dfc9be7c176d46bb,hard_prompt,coding,English,Expand the following LLM prompt to detect tabu...
4,666d2acdd7d64e17,hard_prompt,coding,English,write a script that will generate glowing text...


In [4]:
hard['subcategory'].value_counts()

subcategory
coding    253
math      247
Name: count, dtype: int64

In [5]:
# Language breakdown — surfaced for transparency. We do NOT filter to English;
# all 500 prompts are retained per the locked scope. Multilingual subset is a
# future D-axis variant (cross-language robustness check).
print('Language breakdown of 500 hard prompts:')
print(hard['language'].value_counts())
print(f'\nEnglish: {(hard["language"] == "English").sum()}')
print(f'Non-English: {(hard["language"] != "English").sum()}')

Language breakdown of 500 hard prompts:
language
English       363
Chinese        57
Russian        37
Vietnamese     13
Korean          6
Spanish         6
Japanese        5
French          2
German          2
Indonesian      2
Finnish         2
Turkish         1
Italian         1
Armenian        1
Portuguese      1
Czech           1
Name: count, dtype: int64

English: 363
Non-English: 137


## Sanity check 2 — candidate + baseline responses present

In [6]:
candidate_path = DATA_DIR / 'model_answer' / 'gpt-4.1-mini.jsonl'
baseline_path = DATA_DIR / 'model_answer' / 'o3-mini-2025-01-31.jsonl'

candidate = pd.read_json(candidate_path, lines=True)
baseline = pd.read_json(baseline_path, lines=True)

print(f'Candidate (gpt-4.1-mini): {len(candidate)} responses')
print(f'Baseline (o3-mini-2025-01-31): {len(baseline)} responses')

Candidate (gpt-4.1-mini): 750 responses
Baseline (o3-mini-2025-01-31): 750 responses


In [7]:
# Coverage check — verify every hard UID has both a candidate and baseline response.
# (Each model has 750 responses across all categories; we need the 500 hard ones.)
hard_uids = set(hard['uid'])
candidate_uids = set(candidate['uid'])
baseline_uids = set(baseline['uid'])

missing_candidate = hard_uids - candidate_uids
missing_baseline = hard_uids - baseline_uids

print(f'Hard UIDs missing in candidate: {len(missing_candidate)}')
print(f'Hard UIDs missing in baseline: {len(missing_baseline)}')
assert not missing_candidate, f'Candidate missing {len(missing_candidate)} UIDs'
assert not missing_baseline, f'Baseline missing {len(missing_baseline)} UIDs'
print('✓ Full coverage — all 500 hard prompts have candidate + baseline responses')

Hard UIDs missing in candidate: 0
Hard UIDs missing in baseline: 0
✓ Full coverage — all 500 hard prompts have candidate + baseline responses


## Sanity check 3 — read v2.0 judge config + JUDGE_SETTINGS

Inspect the repo's shipped configuration so the anchor cell exactly matches v2.0 default.

In [8]:
import yaml
with open(REPO_DIR / 'config' / 'arena-hard-v2.0.yaml') as f:
    judge_config = yaml.safe_load(f)
judge_config

{'judge_model': 'gpt-4.1',
 'temperature': 0.0,
 'max_tokens': 16000,
 'bench_name': 'arena-hard-v2.0',
 'reference': None,
 'regex_patterns': ['\\[\\[([AB<>=]+)\\]\\]', '\\[([AB<>=]+)\\]'],
 'prompt_template': "<|User Prompt|>\n{QUESTION}\n\n<|The Start of Assistant A's Answer|>\n{ANSWER_A}\n<|The End of Assistant A's Answer|>\n\n<|The Start of Assistant B's Answer|>\n{ANSWER_B}\n<|The End of Assistant B's Answer|>",
 'model_list': ['deepseek-r1']}

In [9]:
# Read JUDGE_SETTINGS from utils/judge_utils.py
# Per-category baseline mapping — confirm 'hard' category baseline = o3-mini-2025-01-31
judge_utils = (REPO_DIR / 'utils' / 'judge_utils.py').read_text()
print(judge_utils)

OG_ARENA_HARD_PROMPT = "Please act as an impartial judge and evaluate the quality of the responses provided by two AI assistants to the user prompt displayed below. You will be given assistant A's answer and assistant B's answer. Your job is to evaluate which assistant's answer is better.\n\nBegin your evaluation by generating your own answer to the prompt. You must provide your answers before judging any answers.\n\nWhen evaluating the assistants' answers, compare both assistants' answers with your answer. You must identify and correct any mistakes or inaccurate information.\n\nThen consider if the assistant's answers are helpful, relevant, and concise. Helpful means the answer correctly responds to the prompt or follows the instructions. Note when user prompt has any ambiguity or more than one interpretation, it is more helpful and appropriate to ask for clarifications or more information from the user than providing an answer based on assumptions. Relevant means all parts of the res

## Build preprocessed pairs

Join candidate + baseline answers per prompt. Save as one canonical file for all downstream notebooks.

In [10]:
# Extract response text from messages list (last assistant message)
def extract_response(messages):
    """Pull the assistant's response text from the messages array.

    Arena-Hard model_answer files store messages as a list of role/content dicts.
    The assistant content may be a dict {'answer': str} or a plain string.
    """
    for msg in reversed(messages):
        if msg.get('role') == 'assistant':
            content = msg.get('content', '')
            if isinstance(content, dict):
                return content.get('answer', '')
            return content
    return None

candidate['candidate_response'] = candidate['messages'].apply(extract_response)
baseline['baseline_response'] = baseline['messages'].apply(extract_response)

# Join: hard prompts + candidate + baseline on uid
pairs = (
    hard[['uid', 'category', 'subcategory', 'language', 'prompt']]
    .merge(candidate[['uid', 'candidate_response']], on='uid', how='left')
    .merge(baseline[['uid', 'baseline_response']], on='uid', how='left')
)

# Sanity check
assert len(pairs) == 500, f'Expected 500 pairs, got {len(pairs)}'
assert pairs['candidate_response'].notna().all(), 'Some candidate responses missing'
assert pairs['baseline_response'].notna().all(), 'Some baseline responses missing'

# Save canonical preprocessed file
output_path = Path('data/preprocessed_pairs.jsonl')
output_path.parent.mkdir(parents=True, exist_ok=True)
pairs.to_json(output_path, orient='records', lines=True, force_ascii=False)
print(f'Saved {len(pairs)} pairs to {output_path}')
pairs.head()

Saved 500 pairs to data/preprocessed_pairs.jsonl


,uid,category,subcategory,language,prompt,candidate_response,baseline_response
0,2edbb5f36f5b42be,hard_prompt,coding,English,Write me a zig program that solves the followi...,Certainly! Here's a Zig program that reads the...,Below is one acceptable Zig solution. This pro...
1,ec71c09662a64365,hard_prompt,coding,English,please write a python script that takes a .mp4...,Certainly! Below is a Python script that uses ...,Below is an example Python script that uses Op...
2,d5cdf24c4e614beb,hard_prompt,coding,English,"<div style=""width: 100vh; height: 100vh;"">\n ...",To center child elements both vertically and h...,You can use flexbox on the parent element to c...
3,dfc9be7c176d46bb,hard_prompt,coding,English,Expand the following LLM prompt to detect tabu...,Certainly! Here's the expanded prompt that inc...,Below is the expanded version of your LLM prom...
4,666d2acdd7d64e17,hard_prompt,coding,English,write a script that will generate glowing text...,Certainly! Below is a complete example of HTML...,Below is an example of an HTML file with embed...


In [11]:
# Length sanity — useful for cost estimation and seeing if any responses are unusually long.
# Rough approximation: 1 token ≈ 0.75 words.
pairs['prompt_words'] = pairs['prompt'].str.split().str.len()
pairs['candidate_words'] = pairs['candidate_response'].str.split().str.len()
pairs['baseline_words'] = pairs['baseline_response'].str.split().str.len()

print('Word counts (median / 90th percentile / max):')
for col in ['prompt_words', 'candidate_words', 'baseline_words']:
    p50, p90, pmax = pairs[col].quantile([0.5, 0.9, 1.0])
    print(f'  {col}: {int(p50)} / {int(p90)} / {int(pmax)}')

# Total approx tokens per judge call (system prompt + user prompt + both responses)
# OG_ARENA_HARD_PROMPT is ~400 words; prompt template overhead is small.
approx_tokens_per_call = (400 + pairs['prompt_words'] + pairs['candidate_words'] + pairs['baseline_words']) / 0.75
print(f'\nApprox input tokens per judge call (median / 90th / max):')
print(f'  {int(approx_tokens_per_call.median())} / {int(approx_tokens_per_call.quantile(0.9))} / {int(approx_tokens_per_call.max())}')

Word counts (median / 90th percentile / max):
  prompt_words: 56 / 376 / 2706
  candidate_words: 487 / 1091 / 4402
  baseline_words: 407 / 1048 / 2364

Approx input tokens per judge call (median / 90th / max):
  1862 / 3867 / 7937


## Summary

Notebook 01 complete — data loaded, sanity checks passed, preprocessed pairs saved.

### Dataset

- **Source:** Arena-Hard v2.0 (500 hard prompts, filtered from 750 total)
- **Subcategory:** 253 coding + 247 math
- **Language:** 363 English + 137 non-English (no filter applied)

### Locked anchor

- **Judge:** `gpt-4.1` (v2.0 default)
- **Candidate (held fixed at L0):** `gpt-4.1-mini`
- **Baseline (per-category fixed):** `o3-mini-2025-01-31`
- **Rubric (C1):** `OG_ARENA_HARD_PROMPT`
- **Verdict format (F1):** reasoning before verdict, 5-point scale (`[[A>>B]]` / `[[A>B]]` / `[[A=B]]` / `[[B>A]]` / `[[B>>A]]`)
- **Position bias:** two-round bidirectional swap

### Coverage

- ✓ All 500 hard UIDs have candidate responses
- ✓ All 500 hard UIDs have baseline responses

### Output saved

`data/preprocessed_pairs.jsonl` — 500 rows, schema:
`(uid, category, subcategory, language, prompt, candidate_response, baseline_response, prompt_words, candidate_words, baseline_words)`

### Length stats

- Median input tokens per judge call: ~1,862
- 90th percentile: ~3,867
- Max: ~7,937 (well within `max_tokens=16000` budget)

### Next

**Notebook 02 — pilot:** n=20 × 4 corner cells, ~£1, validates pipeline before full factorial. Requires `_pipeline.py` and `_prompts.py` (now present in this directory).